In [11]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

import joblib 
import warnings
warnings.filterwarnings('ignore')

from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity



In [12]:
df = pd.read_csv("../cleaned Datasets/ProductInfo.csv")
metedata = pd.read_csv("../cleaned Datasets/MetaData.csv")



In [13]:
tfidf = TfidfVectorizer(
    stop_words='english' ,
    ngram_range=(1,2), 
    max_features=30000  
)


In [14]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [15]:
def rank_recommendation(recommendations , scores , top_indx):
    recommendations = recommendations.copy()

    recommendations['similarity'] = scores[top_indx]
    recommendations['popularity_norm'] = (
        recommendations['Popularity Score']
        / recommendations['Popularity Score'].max()
    )

    recommendations['value_norm'] = (
        recommendations['Value Score']
        / recommendations['Value Score'].max()
    )

    recommendations['Final Score'] = (
        0.80 * recommendations['similarity']
        + 0.13 * recommendations['popularity_norm']
        + 0.07 * recommendations['value_norm']
    )

    return recommendations.sort_values(
        by='Final Score',
        ascending=False
    )


In [16]:
tfidf_matrix.shape

(66960, 30000)

In [17]:
def recommend(product_id):
    matched_data = df[
        df['product Id'] == product_id
    ]

    if matched_data.empty:
        return pd.DataFrame()
    

    idx = matched_data.index[0] 

    scores = cosine_similarity(
        tfidf_matrix[idx] ,
        tfidf_matrix
    ).flatten()

    top_idx = np.argsort(scores)[::-1][1:51]

    recommendations = df.iloc[top_idx].copy()
    recommendations = rank_recommendation(recommendations , scores , top_idx)

    return recommendations.head(20)


In [18]:
def recommendBykeywords(text):

    query_vector = tfidf.transform([text]) 

    scores = cosine_similarity(
        query_vector ,
        tfidf_matrix
    ).flatten()

    top_idx = np.argsort(scores)[::-1][:50]

    recommendations = df.iloc[top_idx].copy()
    recommendations = rank_recommendation(recommendations , scores , top_idx)
    return recommendations.head(10)


In [21]:
recommendBykeywords('electronics').sample(5)

,name,main_category,sub_category,ratings,no_of_ratings,discount_price,actual_price,product Id,discount percentage,Rating Bucket,Popularity Score,Value Score,cleaned_name,cleaned_sub_category,tags,similarity,popularity_norm,value_norm,Final Score
61083,"Bharat's Original Nusqa E Arabia, 500ml",tv audio camera,All Electronics,4.0,1119,500.00,500.0,P71595,0.000000,Good,28.084336,0.056169,"bharat's nusqa e arabia, 500ml",Electronics,tv audio camera All Electronics Bharat's Origi...,0.265400,0.680632,0.036356,0.303347
42456,"Strauss Skipping Rope, (Grey/Blue)",tv audio camera,All Electronics,4.0,11746,90.49,299.0,P47945,69.735786,Good,37.485413,0.414249,"strauss skipping rope, (greyblue)",Electronics,tv audio camera All Electronics Strauss Skippi...,0.200868,0.908470,0.268126,0.297564
2358,DYMO DYM12966 Organizer Xpress Pro Light Plast...,tv audio camera,All Electronics,4.0,30197,1400.00,1990.0,P2417,29.648241,Good,41.262124,0.029473,dymo dym12966 organizer xpress pro light plast...,Electronics,tv audio camera All Electronics DYMO DYM12966 ...,0.241770,1.000000,0.019077,0.324752
17265,PopSockets Basic PopGrip – Black,tv audio camera,All Electronics,4.0,985,499.00,599.0,P18415,16.694491,Good,27.574625,0.055260,popsockets basic popgrip – black,Electronics,tv audio camera All Electronics PopSockets Bas...,0.299238,0.668279,0.035767,0.328770
17756,Gizga Essentials Professional 6-in-1 Cleaning ...,tv audio camera,All Electronics,4.0,25215,299.00,499.0,P18956,40.080160,Good,40.540936,0.135588,gizga essentials professional 6in1 cleaning ki...,Electronics,tv audio camera All Electronics Gizga Essentia...,0.201942,0.982522,0.087761,0.295424


In [22]:
joblib.dump(tfidf , "../models/tfidf_vectorizer.pkl")
joblib.dump(df , "../models/products.pkl")

sparse.save_npz("../models/tfidf_matrix.npz" , tfidf_matrix)